In [15]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os

from pathlib import Path


In [16]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [17]:
print(os.getcwd())  # Prints the current working directory

/home/david/Personal-Project/olist-data-analytics/notebooks


In [22]:
# Read Raw Data
orders_df = pd.read_csv("../data/raw/orders.csv")
order_items_df = pd.read_csv("../data/raw/order_items.csv")
order_payments_df = pd.read_csv("../data/raw/order_payments.csv")
order_reviews_df = pd.read_csv("../data/raw/order_reviews.csv")
products_df = pd.read_csv("../data/raw/products.csv")
product_category_name_translation_df = pd.read_csv("../data/raw/product_category_name_translation.csv")
sellers_df = pd.read_csv("../data/raw/sellers.csv")
customers_df = pd.read_csv("../data/raw/customers.csv")
geolocation_df = pd.read_csv("../data/raw/geolocation.csv")



In [23]:
# Create a Copy of the Raw Data
orders_df_clean = orders_df.copy()
order_items_df_clean = order_items_df.copy()
order_payments_df_clean = order_payments_df.copy()
order_reviews_df_clean = order_reviews_df.copy()
products_df_clean = products_df.copy()
product_category_name_translation_dfc_lean = product_category_name_translation_df.copy()
sellers_df_clean = sellers_df.copy()
customers_df_clean = customers_df.copy()
geolocation_df_clean = geolocation_df.copy()

In [ ]:
# Initial Data Validation
print(f"ORDERS SHAPE")
orders_df_clean.shape

print(f"\n ORDERS INFO")
orders_df_clean.info()

print(f"\n ORDERS HEAD")
print(orders_df_clean.head(6))

print(f"\n ORDERS DESCRIPTION")
orders_df_clean.describe(include="all")



In [33]:
#standardize column name
orders_df_clean.columns = (
    orders_df_clean.columns
    .str.strip()
    .str.lower()
)

orders_df_clean.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [ ]:
# Fix Data Types
data_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_df_clean[data_cols] = orders_df_clean[data_cols].apply(pd.to_datetime)


# Convert order_status to category column
orders_df_clean["order_status"] = (orders_df_clean["order_status"]).astype("category")

# check the datatype
orders_df_clean.dtypes

In [ ]:
# Remove Duplicate Rows
orders_df_clean.duplicated().sum() # check if duplicate exist
orders_df_clean["order_id"].duplicated().sum() # check if the primary key has a duplicate

if orders_df_clean.duplicated().sum() and orders_df_clean["order_id"].duplicate():
    orders_df_clean = orders_df_clean.drop_duplicates()

In [55]:
# Check for Missing Values
orders_df_clean.isna().sum() # checks the entire data frame to find null or empty cells in any of the columns and return the count value

# orders_df_clean.isna().mean().mul(100).sort_values(ascending=False) # find the percentage of missing values

#create Boolean masks for each timestamp
purchase_exists = orders_df_clean["order_purchase_timestamp"].notna()
approved_exists = orders_df_clean["order_approved_at"].notna()
carrier_exists = orders_df_clean["order_delivered_carrier_date"].notna()
customer_exists = orders_df_clean["order_delivered_customer_date"].notna()
estimated_exists = orders_df_clean["order_estimated_delivery_date"].notna()

# print(f"\npurchase_exists\n{purchase_exists}")
# print(f"\napproved_exists\n{approved_exists}")
# print(f"\ncarrier_exists\n{carrier_exists}")
# print(f"\ncustomer_exists\n{customer_exists}")
# print(f"\nestimated_exists\n{estimated_exists}")

#get the order status
status = orders_df_clean["order_status"]
print(f"ORDER STATUS\n", status)

# Extra
delivered_anomalies = orders_df_clean[
    (status == "delivered") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~carrier_exists |
        ~customer_exists |
        ~estimated_exists
    )
]

shipped_anomalies = orders_df_clean[
    (status == "shipped") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~carrier_exists |
        ~estimated_exists
    )
]

approved_anomalies = orders_df_clean[
    (status == "approved") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~estimated_exists
    )
]

processing_anomalies = orders_df_clean[
    (status == "processing") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~estimated_exists
    )
]

invoiced_anomalies = orders_df_clean[
    (status == "invoiced") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~estimated_exists
    )
]


canceled_anomalies = orders_df_clean[
    (status == "canceled") &
    (
        ~purchase_exists |
        ~estimated_exists |
        carrier_exists |
        customer_exists
    )
]

unavailable_anomalies = orders_df_clean[
    (status == "unavailable") &
    (
        ~purchase_exists |
        ~approved_exists |
        ~estimated_exists
    )
]

anomalies = pd.concat([
    delivered_anomalies,
    shipped_anomalies,
    approved_anomalies,
    processing_anomalies,
    invoiced_anomalies,
    canceled_anomalies,
    unavailable_anomalies
]).drop_duplicates()


len(anomalies)
anomalies.shape

anomalies[
    [
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]


ORDER STATUS
 0        delivered
1        delivered
2        delivered
3        delivered
4        delivered
           ...    
99436    delivered
99437    delivered
99438    delivered
99439    delivered
99440    delivered
Name: order_status, Length: 99441, dtype: category
Categories (8, str): ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']


,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
22663,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
